In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
df = pd.read_csv('../data/raw/all_matches.csv')

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

C:\Users\princ\AppData\Local\Temp\ipykernel_9728\1759151701.py:1: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/all_matches.csv')


Shape: (295732, 27)

Columns:
 ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']

First 5 rows:


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,2,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("\nSeasons available:", df['season'].unique())
print("Total matches:", df['match_id'].nunique())

print("\nMissing values:")
print(df.isnull().sum())

Total rows: 295732
Total columns: 27

Seasons available: [2017 2018 2019 '2019' '2020/21' '2021' 2021 2022 2023 2024 2025 '2025'
 '2026' '2007/08' '2009' '2009/10' '2011' '2012' 2012 2013 2014 2015 2016]
Total matches: 1243

Missing values:
match_id                       0
season                         0
start_date                     0
venue                          0
innings                        0
ball                           0
actual_delivery                0
batting_team                   0
bowling_team                   0
striker                        0
non_striker                    0
bowler                         0
runs_off_bat                   0
extras                         0
wides                     285856
noballs                   294506
byes                      294987
legbyes                   291326
penalty                   295730
non_boundary              295703
wicket_type               281027
player_dismissed          281027
other_wicket_type         295732


In [7]:
# Fix season column - standardise everything to a clean year string
def clean_season(s):
    s = str(s).strip()
    season_map = {
        '2007/08': '2008',
        '2009/10': '2010',
        '2020/21': '2021',
    }
    if s in season_map:
        return season_map[s]
    return s.replace("'", "").strip()

df['season'] = df['season'].apply(clean_season)

# Now confirm seasons are clean
print("Seasons cleaned:", sorted(df['season'].unique()))

# Fill missing extras with 0
extra_cols = ['wides', 'noballs', 'byes', 'legbyes', 'penalty']
df[extra_cols] = df[extra_cols].fillna(0)

# Fill missing wicket info
df['wicket_type'] = df['wicket_type'].fillna('none')
df['player_dismissed'] = df['player_dismissed'].fillna('none')

# Fix team name inconsistencies
team_name_map = {
    'Delhi Daredevils'        : 'Delhi Capitals',
    'Deccan Chargers'         : 'Sunrisers Hyderabad',
    'Kings XI Punjab'         : 'Punjab Kings',
    'Rising Pune Supergiants' : 'Rising Pune Supergiant',
}
df['batting_team'] = df['batting_team'].replace(team_name_map)
df['bowling_team'] = df['bowling_team'].replace(team_name_map)

# Convert date to datetime
df['start_date'] = pd.to_datetime(df['start_date'])

# Extract over number from ball column (e.g. 3.2 → over 3)
df['over_number'] = df['ball'].apply(lambda x: int(str(x).split('.')[0]))

# Flag legal deliveries (not wides or no balls)
df['is_legal'] = ((df['wides'] == 0) & (df['noballs'] == 0)).astype(int)

print("\nCleaning done!")
print("Legal deliveries:", df['is_legal'].sum())
print("Unique teams:", sorted(df['batting_team'].unique()))

Seasons cleaned: ['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2021', '2022', '2023', '2024', '2025', '2026']

Cleaning done!
Legal deliveries: 284630
Unique teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [8]:
# Fix RCB name (same team, city renamed)
df['batting_team'] = df['batting_team'].replace({'Royal Challengers Bengaluru': 'Royal Challengers Bangalore'})
df['bowling_team'] = df['bowling_team'].replace({'Royal Challengers Bengaluru': 'Royal Challengers Bangalore'})

print("Teams final:", sorted(df['batting_team'].unique()))

# Create match phase column
def get_phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 14:
        return 'Middle'
    else:
        return 'Death'

df['phase'] = df['over_number'].apply(get_phase)

# Verify phase distribution
print("\nPhase distribution:")
print(df['phase'].value_counts())

print("\nSample data:")
df[['match_id', 'season', 'striker', 'bowler', 'over_number', 'phase', 'runs_off_bat', 'is_legal']].head(10)

Teams final: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bangalore', 'Sunrisers Hyderabad']

Phase distribution:
phase
Middle       135655
Powerplay     92904
Death         67173
Name: count, dtype: int64

Sample data:


,match_id,season,striker,bowler,over_number,phase,runs_off_bat,is_legal
0,1082591,2017,DA Warner,TS Mills,0,Powerplay,0,1
1,1082591,2017,DA Warner,TS Mills,0,Powerplay,0,1
2,1082591,2017,DA Warner,TS Mills,0,Powerplay,4,1
3,1082591,2017,DA Warner,TS Mills,0,Powerplay,0,1
4,1082591,2017,DA Warner,TS Mills,0,Powerplay,0,0
5,1082591,2017,S Dhawan,TS Mills,0,Powerplay,0,1
6,1082591,2017,S Dhawan,TS Mills,0,Powerplay,0,1
7,1082591,2017,S Dhawan,A Choudhary,1,Powerplay,1,1
8,1082591,2017,DA Warner,A Choudhary,1,Powerplay,4,1
9,1082591,2017,DA Warner,A Choudhary,1,Powerplay,0,0


In [9]:
# Filter only legal deliveries for batting stats
legal = df[df['is_legal'] == 1].copy()

# Group by player and phase
batting = legal.groupby(['striker', 'phase']).agg(
    runs    = ('runs_off_bat', 'sum'),
    balls   = ('is_legal', 'count')
).reset_index()

# Calculate strike rate
batting['strike_rate'] = (batting['runs'] / batting['balls'] * 100).round(2)

# Filter minimum 50 balls per phase (removes players with too few balls)
batting = batting[batting['balls'] >= 50]

# Pivot for easy reading
batting_pivot = batting.pivot_table(
    index   = 'striker',
    columns = 'phase',
    values  = ['runs', 'balls', 'strike_rate']
).round(2)

print("Batting analysis done!")
print("Total qualified batters:", batting['striker'].nunique())
print("\nTop 10 Death over hitters (min 50 balls):")
death_batting = batting[batting['phase'] == 'Death'].sort_values('strike_rate', ascending=False)
print(death_batting[['striker', 'runs', 'balls', 'strike_rate']].head(10))

Batting analysis done!
Total qualified batters: 364

Top 10 Death over hitters (min 50 balls):
              striker  runs  balls  strike_rate
504           HM Amla   120     51       235.29
69     AB de Villiers  1837    829       221.59
1174       RM Patidar   267    124       215.32
763    LS Livingstone   334    156       214.10
1081        Q de Kock   240    116       206.90
227   B Sai Sudharsan   302    146       206.85
808        MA Agarwal   252    122       206.56
328        D Ferreira   192     94       204.26
1184          RR Pant  1084    537       201.86
1396     Sameer Rizvi   131     65       201.54


In [10]:
# Calculate bowler runs conceded per delivery
# Bowler is charged: runs off bat + wides + noballs
df['bowler_runs'] = df['runs_off_bat'] + df['wides'] + df['noballs']

# For bowling, count legal balls per bowler
bowling = df.groupby(['bowler', 'phase']).agg(
    runs_conceded = ('bowler_runs', 'sum'),
    legal_balls   = ('is_legal', 'sum'),
    wickets       = ('wicket_type', lambda x: (x != 'none').sum())
).reset_index()

# Calculate overs and economy
bowling['overs'] = (bowling['legal_balls'] / 6).round(2)
bowling['economy'] = (bowling['runs_conceded'] / bowling['overs']).round(2)

# Filter minimum 60 legal balls per phase
bowling = bowling[bowling['legal_balls'] >= 60]

print("Bowling analysis done!")
print("Total qualified bowlers:", bowling['bowler'].nunique())

# Top death over specialists
print("\nTop 10 Death Over Specialists (lowest economy, min 60 balls):")
death_bowling = bowling[bowling['phase'] == 'Death'].sort_values('economy')
print(death_bowling[['bowler', 'runs_conceded', 'overs', 'wickets', 'economy']].head(10))

# Top powerplay bowlers
print("\nTop 10 Powerplay Bowlers (lowest economy, min 60 balls):")
pp_bowling = bowling[bowling['phase'] == 'Powerplay'].sort_values('economy')
print(pp_bowling[['bowler', 'runs_conceded', 'overs', 'wickets', 'economy']].head(10))

Bowling analysis done!
Total qualified bowlers: 373

Top 10 Death Over Specialists (lowest economy, min 60 balls):
                bowler  runs_conceded   overs  wickets  economy
1335     Sohail Tanvir          117.0   17.17       18     6.81
1258         SP Narine         1336.0  184.17       94     7.25
338       DE Bollinger          292.0   39.00       28     7.49
20            A Kumble          212.0   27.83       18     7.62
1441         VY Mahesh           80.0   10.50       11     7.62
508            J Botha          136.0   17.67        9     7.70
416            GB Hogg          119.0   15.33        7     7.76
1236        SL Malinga         1464.0  186.17      122     7.86
893           N Burger           83.0   10.50        6     7.90
1077  RE van der Merwe          125.0   15.83        7     7.90

Top 10 Powerplay Bowlers (lowest economy, min 60 balls):
               bowler  runs_conceded  overs  wickets  economy
90         AG Murtaza           52.0   13.0        3     4.00

In [11]:
# Load match info from all_matches to get toss data
# Toss info is in the main df - extract unique match level data
match_info = df.groupby('match_id').agg(
    season      = ('season', 'first'),
    venue       = ('venue', 'first'),
    team1       = ('batting_team', 'first'),
).reset_index()

# Get innings scores
innings_scores = df.groupby(['match_id', 'innings']).agg(
    total_runs  = ('runs_off_bat', 'sum'),
    extras      = ('extras', 'sum'),
    batting_team = ('batting_team', 'first')
).reset_index()

innings_scores['total'] = innings_scores['total_runs'] + innings_scores['extras']

# First innings scores
first_innings = innings_scores[innings_scores['innings'] == 1][['match_id', 'batting_team', 'total']].rename(
    columns={'batting_team': 'first_bat_team', 'total': 'first_innings_score'})

# Second innings scores
second_innings = innings_scores[innings_scores['innings'] == 2][['match_id', 'batting_team', 'total']].rename(
    columns={'batting_team': 'second_bat_team', 'total': 'second_innings_score'})

# Merge
match_scores = first_innings.merge(second_innings, on='match_id', how='inner')
match_scores['winner'] = match_scores.apply(
    lambda x: x['first_bat_team'] if x['first_innings_score'] > x['second_innings_score'] else x['second_bat_team'],
    axis=1
)

print("Match scores calculated!")
print("Total matches with result:", len(match_scores))
print("\nSample:")
print(match_scores.head())

Match scores calculated!
Total matches with result: 1237

Sample:
   match_id         first_bat_team  first_innings_score  \
0    335982  Kolkata Knight Riders                  222   
1    335983    Chennai Super Kings                  240   
2    335984       Rajasthan Royals                  129   
3    335985         Mumbai Indians                  165   
4    335986    Sunrisers Hyderabad                  110   

               second_bat_team  second_innings_score  \
0  Royal Challengers Bangalore                    82   
1                 Punjab Kings                   207   
2               Delhi Capitals                   132   
3  Royal Challengers Bangalore                   166   
4        Kolkata Knight Riders                   112   

                        winner  
0        Kolkata Knight Riders  
1          Chennai Super Kings  
2               Delhi Capitals  
3  Royal Challengers Bangalore  
4        Kolkata Knight Riders  


In [12]:
# Load toss data directly from cricsheet all_matches info
# We'll extract toss info from df itself - who batted first = toss winner chose to bat
# Let's build toss analysis from match_scores

# Which team won by chasing vs batting first
match_scores['result_type'] = match_scores.apply(
    lambda x: 'Bat First Win' if x['winner'] == x['first_bat_team'] else 'Chase Win',
    axis=1
)

# Overall bat first vs chase win %
result_counts = match_scores['result_type'].value_counts()
print("Overall Results:")
print(result_counts)
print(f"\nBat First Win %: {result_counts['Bat First Win'] / len(match_scores) * 100:.1f}%")
print(f"Chase Win %:     {result_counts['Chase Win'] / len(match_scores) * 100:.1f}%")

# Merge venue info
match_scores = match_scores.merge(
    df.groupby('match_id')['venue'].first().reset_index(),
    on='match_id'
)

# Venue wise bat first vs chase win %
venue_analysis = match_scores.groupby(['venue', 'result_type']).size().unstack(fill_value=0)
venue_analysis.columns = ['Bat First Win', 'Chase Win']
venue_analysis['total_matches'] = venue_analysis['Bat First Win'] + venue_analysis['Chase Win']
venue_analysis['bat_first_win_pct'] = (venue_analysis['Bat First Win'] / venue_analysis['total_matches'] * 100).round(1)

# Filter venues with minimum 10 matches
venue_analysis = venue_analysis[venue_analysis['total_matches'] >= 10].sort_values('bat_first_win_pct', ascending=False)

print("\nTop venues favouring Batting First:")
print(venue_analysis[['Bat First Win', 'Chase Win', 'total_matches', 'bat_first_win_pct']].head(8))

print("\nTop venues favouring Chasing:")
print(venue_analysis[['Bat First Win', 'Chase Win', 'total_matches', 'bat_first_win_pct']].tail(8))

Overall Results:
result_type
Chase Win        668
Bat First Win    569
Name: count, dtype: int64

Bat First Win %: 46.0%
Chase Win %:     54.0%

Top venues favouring Batting First:
                                                    Bat First Win  Chase Win  \
venue                                                                          
Maharashtra Cricket Association Stadium, Pune                  10          3   
MA Chidambaram Stadium, Chepauk                                30         18   
Subrata Roy Sahara Stadium                                     10          6   
Kingsmead                                                       9          6   
Brabourne Stadium                                               6          4   
Eden Gardens, Kolkata                                          16         13   
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Sta...              7          6   
Rajiv Gandhi International Stadium                              8          7   

                  